# Part III — Neural Networks in PyTorch

**Exercises from *Machine Learning: The Working Text*.**

Numerically stable softmax, the p − y gradient, convolution by im2col and its arithmetic, scaled dot-product attention, and a PyTorch training step.

---

## How to use this notebook

1. Run the **Setup** cell below once.
2. Work down the notebook. Each exercise has a stub cell — replace the
   `raise NotImplementedYet(...)` with your own code and run it.
3. Run the `check(...)` cell under it. It prints `PASS`, `FAIL` or `TODO`.
4. `check_all()` at the bottom summarises the notebook.

**A `TODO` is never a pass.** The gate thresholds come from the book, so
when one fails the useful question is which of the two is wrong — your
code, or your understanding of the threshold.

Nothing needs installing: Colab already has NumPy and PyTorch. Nothing
is downloaded, and no file is read from disk.

### Exercises in this notebook

| id | exercise | read first |
|---|---|---|
| `17.1` | Numerically stable softmax | Ch 17 · PTG p.37 |
| `17.2` | Cross-entropy gradient | Ch 17 · PTG p.40 |
| `19.1` | Convolution via im2col | Ch 19 · PTG p.128 |
| `19.2` | Convolution output arithmetic | Ch 19 · PTG p.129 |
| `21.1` | Scaled dot-product attention | Ch 21 · PTG p.320 |
| `18.1` | A training step that actually trains | Ch 18 · PTG p.62 |

In [ ]:
#@title Setup — run this cell first (collapsed: it holds the gates and the reference solutions)

import math
import time
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np

try:
    import torch
    _HAS_TORCH = True
except ImportError:
    _HAS_TORCH = False


class NotImplementedYet(NotImplementedError):
    """Raised by a stub you have not filled in yet."""


def _rng(seed: int = 0) -> np.random.Generator:
    """Returns a seeded generator so every gate is reproducible."""
    return np.random.default_rng(seed)


def _fit_predict(x_train, y_train, x_test):
    """Least-squares probe used by the leakage gates."""
    design = np.hstack([x_train, np.ones((x_train.shape[0], 1))])
    weights, *_ = np.linalg.lstsq(design, y_train, rcond=None)
    return np.hstack([x_test, np.ones((x_test.shape[0], 1))]) @ weights


def _r2(y_true, y_pred):
    """Coefficient of determination."""
    total = float(np.sum((y_true - y_true.mean()) ** 2))
    return 1.0 - float(np.sum((y_true - y_pred) ** 2)) / total


# Reference implementations. Two gates compare against these, so
# they have to be here. Reading them before you have tried is a
# way of not learning the material.
class Solutions:
    """Reference implementations of every exercise."""

    @staticmethod
    def lp_norm(x: np.ndarray, p: float) -> float:
        """Returns the l_p norm of a vector."""
        x = np.asarray(x, dtype=np.float64)
        if np.isinf(p):
            return float(np.max(np.abs(x)))
        return float(np.sum(np.abs(x) ** p) ** (1.0 / p))

    @staticmethod
    def gram_schmidt_classical(a: np.ndarray) -> np.ndarray:
        """Orthonormalises columns, projecting against the original column."""
        a = np.asarray(a, dtype=np.float64)
        n, k = a.shape
        q = np.zeros((n, k))
        for j in range(k):
            v = a[:, j].copy()
            for i in range(j):
                v = v - (q[:, i] @ a[:, j]) * q[:, i]
            norm = np.linalg.norm(v)
            if norm < 1e-300:
                raise ValueError(f'column {j} is dependent')
            q[:, j] = v / norm
        return q

    @staticmethod
    def gram_schmidt_modified(a: np.ndarray) -> np.ndarray:
        """Orthonormalises columns, projecting against the running residual."""
        a = np.asarray(a, dtype=np.float64)
        n, k = a.shape
        q = np.zeros((n, k))
        for j in range(k):
            v = a[:, j].copy()
            for i in range(j):
                v = v - (q[:, i] @ v) * q[:, i]
            norm = np.linalg.norm(v)
            if norm < 1e-300:
                raise ValueError(f'column {j} is dependent')
            q[:, j] = v / norm
        return q

    @staticmethod
    def projection_matrix(v: np.ndarray) -> np.ndarray:
        """Returns the matrix projecting onto span(v)."""
        v = np.asarray(v, dtype=np.float64).reshape(-1, 1)
        return (v @ v.T) / float((v.T @ v).item())

    @staticmethod
    def lu_decompose(a: np.ndarray) -> Tuple[np.ndarray, np.ndarray,
                                             np.ndarray]:
        """Factors a square matrix as PA = LU with partial pivoting."""
        a = np.asarray(a, dtype=np.float64)
        n = a.shape[0]
        u = a.copy()
        l = np.eye(n)
        p = np.eye(n)
        for k in range(n - 1):
            pivot = int(np.argmax(np.abs(u[k:, k]))) + k
            if abs(u[pivot, k]) < 1e-14:
                raise ValueError('singular to working precision')
            if pivot != k:
                u[[k, pivot], k:] = u[[pivot, k], k:]
                l[[k, pivot], :k] = l[[pivot, k], :k]
                p[[k, pivot], :] = p[[pivot, k], :]
            for row in range(k + 1, n):
                factor = u[row, k] / u[k, k]
                l[row, k] = factor
                u[row, k:] -= factor * u[k, k:]
        return p, l, u

    @staticmethod
    def lu_solve(p: np.ndarray, l: np.ndarray, u: np.ndarray,
                 b: np.ndarray) -> np.ndarray:
        """Solves Ax = b given PA = LU."""
        pb = p @ np.asarray(b, dtype=np.float64)
        n = len(pb)
        y = np.zeros(n)
        for i in range(n):
            y[i] = pb[i] - l[i, :i] @ y[:i]
        x = np.zeros(n)
        for i in range(n - 1, -1, -1):
            x[i] = (y[i] - u[i, i + 1:] @ x[i + 1:]) / u[i, i]
        return x

    @staticmethod
    def det_via_lu(a: np.ndarray) -> float:
        """Returns det(A) from its LU factorisation."""
        p, _, u = Solutions.lu_decompose(a)
        swaps = int(round(np.sum(np.abs(p - np.eye(p.shape[0]))) / 2.0))
        return float(((-1.0) ** swaps) * np.prod(np.diag(u)))

    @staticmethod
    def power_iteration(a: np.ndarray) -> Tuple[float, np.ndarray]:
        """Returns the dominant eigenpair of a symmetric matrix."""
        a = np.asarray(a, dtype=np.float64)
        rng = np.random.default_rng(0)
        v = rng.normal(size=a.shape[0])
        v /= np.linalg.norm(v)
        for _ in range(10_000):
            w = a @ v
            norm = np.linalg.norm(w)
            if norm < 1e-300:
                return 0.0, v
            w /= norm
            if min(np.linalg.norm(w - v), np.linalg.norm(w + v)) < 1e-14:
                v = w
                break
            v = w
        return float(v @ a @ v), v

    @staticmethod
    def svd_lowrank(a: np.ndarray, k: int) -> np.ndarray:
        """Returns the best rank-k approximation."""
        u, s, vt = np.linalg.svd(np.asarray(a, dtype=np.float64),
                                 full_matrices=False)
        return (u[:, :k] * s[:k]) @ vt[:k, :]

    @staticmethod
    def pca(x: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
        """Returns (components, explained_variance_ratio)."""
        x = np.asarray(x, dtype=np.float64)
        centred = x - x.mean(axis=0)
        _, s, vt = np.linalg.svd(centred, full_matrices=False)
        variance = s ** 2 / (x.shape[0] - 1)
        return vt[:k], variance[:k] / variance.sum()

    @staticmethod
    def central_difference(f: Callable, x: float, h: float) -> float:
        """Approximates f'(x) by central difference."""
        return float((f(x + h) - f(x - h)) / (2.0 * h))

    @staticmethod
    def gradient_descent(grad: Callable, x0: np.ndarray, lr: float,
                         iters: int, momentum: float = 0.0) -> np.ndarray:
        """Minimises a function given its gradient."""
        x = np.asarray(x0, dtype=np.float64).copy()
        velocity = np.zeros_like(x)
        for _ in range(iters):
            g = np.asarray(grad(x), dtype=np.float64)
            velocity = momentum * velocity - lr * g
            x = x + velocity
            if not np.all(np.isfinite(x)):
                break
        return x

    @staticmethod
    def numeric_jacobian(f: Callable, x: np.ndarray,
                         h: float = 1e-6) -> np.ndarray:
        """Approximates the Jacobian of f at x."""
        x = np.asarray(x, dtype=np.float64)
        base = np.atleast_1d(np.asarray(f(x), dtype=np.float64))
        jac = np.zeros((base.size, x.size))
        for i in range(x.size):
            step = np.zeros_like(x)
            step[i] = h
            plus = np.atleast_1d(np.asarray(f(x + step), dtype=np.float64))
            minus = np.atleast_1d(np.asarray(f(x - step), dtype=np.float64))
            jac[:, i] = (plus - minus) / (2.0 * h)
        return jac

    @staticmethod
    def two_layer_backward(w1: np.ndarray, b1: np.ndarray, w2: np.ndarray,
                           b2: np.ndarray, x: np.ndarray,
                           y: float) -> Tuple[float, Dict[str, np.ndarray]]:
        """Forward and backward pass of a two-layer tanh network."""
        z1 = w1 @ x + b1
        a1 = np.tanh(z1)
        z2 = w2 @ a1 + b2
        loss = 0.5 * float(((z2 - y) ** 2).item())
        d2 = (z2 - y)
        grads = {'w2': np.outer(d2, a1), 'b2': d2}
        d1 = (w2.T @ d2) * (1.0 - a1 ** 2)
        grads['w1'] = np.outer(d1, x)
        grads['b1'] = d1
        return loss, grads

    @staticmethod
    def shell_fraction(d: int, eps: float) -> float:
        """Fraction of a unit ball's volume within eps of the surface."""
        return float(1.0 - (1.0 - eps) ** d)

    @staticmethod
    def entropy(p: np.ndarray) -> float:
        """Shannon entropy in nats."""
        p = np.asarray(p, dtype=np.float64)
        nonzero = p[p > 0]
        return float(-np.sum(nonzero * np.log(nonzero)))

    @staticmethod
    def kl_divergence(p: np.ndarray, q: np.ndarray) -> float:
        """KL divergence from q to p, in nats."""
        p = np.asarray(p, dtype=np.float64)
        q = np.asarray(q, dtype=np.float64)
        mask = p > 0
        if np.any(q[mask] <= 0):
            raise ValueError('KL is infinite where q = 0 and p > 0')
        return float(np.sum(p[mask] * np.log(p[mask] / q[mask])))

    @staticmethod
    def mle_gaussian(samples: np.ndarray) -> Tuple[float, float]:
        """Maximum-likelihood mean and variance."""
        x = np.asarray(samples, dtype=np.float64)
        mu = float(x.mean())
        return mu, float(((x - mu) ** 2).mean())

    @staticmethod
    def welford(values: np.ndarray) -> Tuple[float, float]:
        """One-pass stable mean and sample variance."""
        count, mean, m2 = 0, 0.0, 0.0
        for value in np.asarray(values, dtype=np.float64).ravel():
            count += 1
            delta = value - mean
            mean += delta / count
            m2 += delta * (value - mean)
        return mean, (m2 / (count - 1) if count > 1 else 0.0)

    @staticmethod
    def bayes_posterior(sensitivity: float, specificity: float,
                        prevalence: float) -> float:
        """P(disease | positive test)."""
        true_pos = sensitivity * prevalence
        false_pos = (1.0 - specificity) * (1.0 - prevalence)
        return float(true_pos / (true_pos + false_pos))

    @staticmethod
    def select_features(x_train: np.ndarray, y_train: np.ndarray,
                        k: int) -> np.ndarray:
        """Indices of the k training-correlated features."""
        centred = y_train - y_train.mean()
        scores = np.abs(x_train.T @ centred)
        return np.argsort(scores)[-k:]

    @staticmethod
    def assert_no_lookforward(folds) -> None:
        """Raises AssertionError if a fold leaks the future."""
        for i, (train_idx, test_idx) in enumerate(folds):
            if len(train_idx) == 0 or len(test_idx) == 0:
                raise AssertionError(f'fold {i}: empty split')
            if int(np.max(train_idx)) >= int(np.min(test_idx)):
                raise AssertionError(f'fold {i}: lookforward leak')

    @staticmethod
    def standardize_fit(x_train: np.ndarray) -> Tuple[np.ndarray,
                                                      np.ndarray]:
        """Returns (mean, std) from the training rows only."""
        x_train = np.asarray(x_train, dtype=np.float64)
        std = x_train.std(axis=0)
        std[std == 0] = 1.0
        return x_train.mean(axis=0), std

    @staticmethod
    def ridge_fit(x: np.ndarray, y: np.ndarray, alpha: float) -> np.ndarray:
        """Ridge coefficients with an unpenalised intercept, placed last."""
        if alpha < 0:
            raise ValueError('alpha must be non-negative')
        x = np.asarray(x, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64)
        x_mean, y_mean = x.mean(axis=0), y.mean()
        xc, yc = x - x_mean, y - y_mean
        d = x.shape[1]
        w = np.linalg.solve(xc.T @ xc + alpha * np.eye(d), xc.T @ yc)
        return np.append(w, y_mean - x_mean @ w)

    @staticmethod
    def logistic_gradient(w: np.ndarray, x: np.ndarray,
                          y: np.ndarray) -> np.ndarray:
        """Gradient of mean binary cross-entropy with respect to w."""
        z = x @ w
        p = 1.0 / (1.0 + np.exp(-z))
        return x.T @ (p - y) / len(y)

    @staticmethod
    def gini_impurity(labels: np.ndarray) -> float:
        """Gini impurity of a label vector."""
        labels = np.asarray(labels)
        if labels.size == 0:
            return 0.0
        _, counts = np.unique(labels, return_counts=True)
        p = counts / labels.size
        return float(1.0 - np.sum(p ** 2))

    @staticmethod
    def precision_recall_f1(y_true: np.ndarray,
                            y_pred: np.ndarray) -> Tuple[float, float,
                                                         float]:
        """Binary precision, recall and F1 for the positive class."""
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        tp = int(np.sum((y_true == 1) & (y_pred == 1)))
        fp = int(np.sum((y_true == 0) & (y_pred == 1)))
        fn = int(np.sum((y_true == 1) & (y_pred == 0)))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        denom = precision + recall
        f1 = 2 * precision * recall / denom if denom else 0.0
        return precision, recall, f1

    @staticmethod
    def bootstrap_ci(metric_fn: Callable, y_true: np.ndarray,
                     y_pred: np.ndarray, n_boot: int, level: float,
                     seed: int) -> Tuple[float, float]:
        """Percentile bootstrap confidence interval."""
        rng = np.random.default_rng(seed)
        n = len(y_true)
        samples = np.empty(n_boot)
        for i in range(n_boot):
            idx = rng.integers(0, n, size=n)
            samples[i] = metric_fn(y_true[idx], y_pred[idx])
        tail = (1.0 - level) / 2.0
        return (float(np.quantile(samples, tail)),
                float(np.quantile(samples, 1.0 - tail)))

    @staticmethod
    def kmeans(x: np.ndarray, k: int, iters: int,
               seed: int) -> Tuple[np.ndarray, np.ndarray, List[float]]:
        """Lloyd's algorithm with a fixed seed."""
        x = np.asarray(x, dtype=np.float64)
        rng = np.random.default_rng(seed)
        centroids = x[rng.choice(len(x), size=k, replace=False)].copy()
        history: List[float] = []
        labels = np.zeros(len(x), dtype=int)
        for _ in range(iters):
            distances = ((x[:, None, :] - centroids[None, :, :]) ** 2).sum(-1)
            labels = np.argmin(distances, axis=1)
            history.append(float(distances[np.arange(len(x)), labels].sum()))
            for j in range(k):
                members = x[labels == j]
                if len(members):
                    centroids[j] = members.mean(axis=0)
        return labels, centroids, history

    @staticmethod
    def stable_softmax(z: np.ndarray) -> np.ndarray:
        """Softmax over the last axis, overflow-safe."""
        z = np.asarray(z, dtype=np.float64)
        shifted = z - z.max(axis=-1, keepdims=True)
        exp = np.exp(shifted)
        return exp / exp.sum(axis=-1, keepdims=True)

    @staticmethod
    def cross_entropy_grad(logits: np.ndarray,
                           one_hot: np.ndarray) -> np.ndarray:
        """Gradient of cross-entropy with respect to the logits."""
        return Solutions.stable_softmax(logits) - one_hot

    @staticmethod
    def conv2d(x: np.ndarray, kernel: np.ndarray) -> np.ndarray:
        """Valid single-channel 2-D cross-correlation."""
        x = np.asarray(x, dtype=np.float64)
        kernel = np.asarray(kernel, dtype=np.float64)
        kh, kw = kernel.shape
        out_h, out_w = x.shape[0] - kh + 1, x.shape[1] - kw + 1
        patches = np.lib.stride_tricks.sliding_window_view(x, (kh, kw))
        cols = patches.reshape(out_h * out_w, kh * kw)
        return (cols @ kernel.ravel()).reshape(out_h, out_w)

    @staticmethod
    def attention(q: np.ndarray, k: np.ndarray,
                  v: np.ndarray) -> np.ndarray:
        """Scaled dot-product attention."""
        scores = (q @ k.T) / np.sqrt(k.shape[-1])
        return Solutions.stable_softmax(scores) @ v

    @staticmethod
    def torch_training_step(model, xb, yb, optimiser, loss_fn) -> float:
        """One zero_grad / forward / backward / step."""
        optimiser.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimiser.step()
        # .detach() first: converting a tensor that still carries
        # requires_grad straight to a float warns, and in a loop it
        # quietly keeps the graph alive.
        return float(loss.detach())

    @staticmethod
    def acf(series: np.ndarray, max_lag: int) -> np.ndarray:
        """Sample autocorrelation for lags 0..max_lag."""
        y = np.asarray(series, dtype=np.float64)
        y = y - y.mean()
        denom = float(y @ y)
        return np.array([float(y[k:] @ y[:len(y) - k]) / denom
                         for k in range(max_lag + 1)])

    @staticmethod
    def additive_decompose(series: np.ndarray,
                           period: int) -> Tuple[np.ndarray, np.ndarray,
                                                 np.ndarray]:
        """Centred-moving-average additive decomposition."""
        y = np.asarray(series, dtype=np.float64)
        kernel = np.ones(period) / period
        trend = np.convolve(y, kernel, mode='same')
        half = period // 2
        trend[:half] = trend[half]
        trend[len(y) - half:] = trend[len(y) - half - 1]
        detrended = y - trend
        seasonal_means = np.array([detrended[i::period].mean()
                                   for i in range(period)])
        seasonal_means -= seasonal_means.mean()
        seasonal = np.array([seasonal_means[i % period]
                             for i in range(len(y))])
        return trend, seasonal, y - trend - seasonal

    @staticmethod
    def rolling_window_cv(n: int, n_folds: int, horizon: int,
                          window: int) -> List[Tuple[np.ndarray,
                                                     np.ndarray]]:
        """Rolling-origin folds with a constant training length."""
        needed = window + n_folds * horizon
        if needed > n:
            raise ValueError(f'need {needed} points, have {n}')
        folds = []
        for i in range(n_folds):
            end = window + i * horizon
            folds.append((np.arange(end - window, end),
                          np.arange(end, end + horizon)))
        return folds

    @staticmethod
    def naive_scale(y_train: np.ndarray, seasonality: int) -> float:
        """MAE of the seasonal naive forecast on the training series."""
        y = np.asarray(y_train, dtype=np.float64)
        if y.size <= seasonality:
            raise ValueError('training series too short')
        scale = float(np.mean(np.abs(y[seasonality:] - y[:-seasonality])))
        if scale == 0.0:
            raise ValueError('constant training series; MASE undefined')
        return scale

    @staticmethod
    def mase(y_true: np.ndarray, y_pred: np.ndarray, y_train: np.ndarray,
             seasonality: int) -> float:
        """Mean absolute scaled error."""
        mae = float(np.mean(np.abs(np.asarray(y_true, dtype=np.float64) -
                                   np.asarray(y_pred, dtype=np.float64))))
        return mae / Solutions.naive_scale(y_train, seasonality)

    @staticmethod
    def absolute_percentage_error(y_true: np.ndarray,
                                  y_pred: np.ndarray) -> np.ndarray:
        """Element-wise APE, undefined at zero targets."""
        y_true = np.asarray(y_true, dtype=np.float64)
        y_pred = np.asarray(y_pred, dtype=np.float64)
        if np.any(y_true == 0.0):
            raise ValueError('APE is undefined for zero targets [TSP p.144]')
        return np.abs((y_true - y_pred) / y_true)

    @staticmethod
    def split_conformal(residuals: np.ndarray, alpha: float) -> float:
        """Conformal half-width for 1 - alpha coverage."""
        s = np.sort(np.asarray(residuals, dtype=np.float64))
        n = s.size
        k = int(np.ceil((n + 1) * (1.0 - alpha))) - 1
        return float(s[min(max(k, 0), n - 1)])

    @staticmethod
    def eigenvalues_2x2(a: np.ndarray) -> Tuple[float, float]:
        """Eigenvalues of a 2x2 from the characteristic polynomial."""
        a = np.asarray(a, dtype=np.float64)
        trace = float(a[0, 0] + a[1, 1])
        det = float(a[0, 0] * a[1, 1] - a[0, 1] * a[1, 0])
        disc = trace ** 2 - 4.0 * det
        if disc < 0:
            raise ValueError('complex eigenvalues; this exercise is real-only')
        root = float(np.sqrt(disc))
        return (trace + root) / 2.0, (trace - root) / 2.0

    @staticmethod
    def singular_values(a: np.ndarray) -> np.ndarray:
        """Singular values as the square roots of the eigenvalues of AᵀA."""
        a = np.asarray(a, dtype=np.float64)
        eigenvalues = np.linalg.eigvalsh(a.T @ a)
        clipped = np.clip(eigenvalues, 0.0, None)
        return np.sqrt(clipped)[::-1]

    @staticmethod
    def optimal_threshold(cost_fp: float, cost_fn: float) -> float:
        """Decision threshold balancing the two expected costs."""
        if cost_fp < 0 or cost_fn < 0 or (cost_fp + cost_fn) == 0:
            raise ValueError('costs must be non-negative and not both zero')
        return float(cost_fp / (cost_fp + cost_fn))

    @staticmethod
    def soft_threshold(c: float, alpha: float) -> float:
        """The 1-D lasso solution."""
        return float(np.sign(c) * max(abs(c) - alpha, 0.0))

    @staticmethod
    def gini_gain(parent: np.ndarray, left: np.ndarray,
                  right: np.ndarray) -> float:
        """Weighted Gini impurity reduction of a split."""
        n = len(parent)
        if n == 0 or len(left) + len(right) != n:
            raise ValueError('children must partition the parent')
        weighted = (len(left) / n) * Solutions.gini_impurity(left) + \
                   (len(right) / n) * Solutions.gini_impurity(right)
        return float(Solutions.gini_impurity(parent) - weighted)

    @staticmethod
    def conv_output_size(in_size: int, kernel: int, stride: int,
                         padding: int, dilation: int) -> int:
        """Convolution output size along one axis."""
        effective = dilation * (kernel - 1)
        return int((in_size + 2 * padding - effective - 1) // stride) + 1

    @staticmethod
    def ar1_acf(phi: float, max_lag: int) -> np.ndarray:
        """Theoretical ACF of an AR(1)."""
        if abs(phi) >= 1.0:
            raise ValueError('|phi| must be < 1 for a stationary AR(1)')
        return np.array([phi ** k for k in range(max_lag + 1)])

    @staticmethod
    def winkler_score(y: float, lower: float, upper: float,
                      alpha: float) -> float:
        """Interval score: width plus an out-of-interval penalty."""
        width = upper - lower
        if y < lower:
            return float(width + (2.0 / alpha) * (lower - y))
        if y > upper:
            return float(width + (2.0 / alpha) * (y - upper))
        return float(width)


# ---- gates ----------------------------------------------------
def _g_softmax(impl) -> str:
    z = np.array([1000.0, 1001.0, 1002.0])
    p = impl.stable_softmax(z)
    assert np.all(np.isfinite(p)), 'softmax overflowed at z ~ 1000'
    assert abs(float(p.sum()) - 1.0) < 1e-12, 'probabilities must sum to 1'
    shifted = impl.stable_softmax(z - 1000.0)
    assert np.allclose(p, shifted, atol=1e-12), 'must be shift-invariant'
    return 'finite at z=1000, sums to 1, shift-invariant'


def _g_ce_grad(impl) -> str:
    rng = _rng(26)
    logits = rng.normal(size=5)
    one_hot = np.zeros(5)
    one_hot[2] = 1.0

    def loss(z):
        shifted = z - z.max()
        return float(-np.log(np.exp(shifted)[2] / np.exp(shifted).sum()))

    analytic = impl.cross_entropy_grad(logits, one_hot)
    numeric = np.zeros(5)
    h = 1e-6
    for i in range(5):
        up, down = logits.copy(), logits.copy()
        up[i] += h
        down[i] -= h
        numeric[i] = (loss(up) - loss(down)) / (2 * h)
    err = float(np.abs(analytic - numeric).max())
    assert err < 1e-6, f'gradient differs by {err:.2e}'
    return f'p - y matches the numeric gradient to {err:.1e}'


def _g_conv(impl) -> str:
    rng = _rng(27)
    x = rng.normal(size=(12, 12))
    kernel = rng.normal(size=(3, 3))
    got = impl.conv2d(x, kernel)
    want = np.zeros((10, 10))
    for i in range(10):
        for j in range(10):
            want[i, j] = float((x[i:i + 3, j:j + 3] * kernel).sum())
    err = float(np.abs(got - want).max())
    assert err < 1e-10, f'differs from the direct loop by {err:.2e}'
    return f'im2col matches the direct convolution to {err:.1e}'


def _g_conv_size(impl) -> str:
    assert impl.conv_output_size(32, 3, 1, 1, 1) == 32, 'same-padding case'
    assert impl.conv_output_size(32, 3, 2, 1, 1) == 16, 'stride-2 case'
    assert impl.conv_output_size(28, 5, 1, 0, 1) == 24, 'valid case'
    assert impl.conv_output_size(32, 3, 1, 2, 2) == 32, 'dilated case'
    return '32->32 (same), 32->16 (stride 2), 28->24 (valid), dilation handled'


def _g_attention(impl) -> str:
    rng = _rng(28)
    q, k, v = (rng.normal(size=(4, 8)), rng.normal(size=(6, 8)),
               rng.normal(size=(6, 3)))
    got = impl.attention(q, k, v)
    scores = (q @ k.T) / np.sqrt(8)
    shifted = scores - scores.max(axis=-1, keepdims=True)
    weights = np.exp(shifted) / np.exp(shifted).sum(axis=-1, keepdims=True)
    want = weights @ v
    assert got.shape == (4, 3), f'shape {got.shape}, expected (4, 3)'
    err = float(np.abs(got - want).max())
    assert err < 1e-10, f'differs by {err:.2e}'
    assert np.allclose(weights.sum(axis=-1), 1.0), 'rows must sum to 1'
    return f'matches the reference to {err:.1e}; attention rows sum to 1'


def _g_training_step(impl) -> str:
    torch.manual_seed(0)
    model = torch.nn.Linear(3, 1)
    optimiser = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = torch.nn.MSELoss()
    xb = torch.randn(32, 3)
    yb = (xb @ torch.tensor([[1.0], [-2.0], [0.5]])) + 0.3
    first = impl.torch_training_step(model, xb, yb, optimiser, loss_fn)
    for _ in range(100):
        last = impl.torch_training_step(model, xb, yb, optimiser, loss_fn)
    assert last < first, f'loss did not fall: {first:.4f} -> {last:.4f}'
    assert last < 0.05, f'loss only reached {last:.4f}'
    return f'loss {first:.4f} -> {last:.5f} in 100 steps'

# ---- exercise harness -------------------------------------------------
# `check('1.1')` runs that exercise's gate against whatever you have
# defined in this notebook. TODO means the function is missing or still
# raises; it is never a pass.

def _impl_namespace():
    """Exposes the notebook's own functions to the gate functions."""

    class _Namespace:
        def __getattr__(self, name):
            value = globals().get(name)
            if callable(value):
                return value
            raise NotImplementedYet(f'{name} is not defined yet')

    return _Namespace()


_GATES = {}


def check(key):
    """Runs one exercise's gate and prints the verdict."""
    if key not in _GATES:
        raise KeyError(f'no exercise {key!r} in this notebook')
    title, gate, needs_torch = _GATES[key]
    if needs_torch and not _HAS_TORCH:
        print(f'[SKIP] {key}  {title}')
        print('       PyTorch is not available in this runtime.')
        return
    try:
        detail = gate(_impl_namespace())
        print(f'[PASS] {key}  {title}')
        if detail:
            print(f'       {detail}')
    except NotImplementedYet as todo:
        print(f'[TODO] {key}  {title}')
        print(f'       not implemented yet: {todo}')
    except AssertionError as failure:
        print(f'[FAIL] {key}  {title}')
        print(f'       {failure or "gate failed"}')
    except Exception as error:                      # noqa: BLE001
        print(f'[FAIL] {key}  {title}')
        print(f'       {type(error).__name__}: {error}')


def check_all():
    """Runs every gate in this notebook and prints a summary."""
    counts = {'PASS': 0, 'FAIL': 0, 'TODO': 0, 'SKIP': 0}
    for key, (title, gate, needs_torch) in _GATES.items():
        if needs_torch and not _HAS_TORCH:
            counts['SKIP'] += 1
            continue
        try:
            gate(_impl_namespace())
            counts['PASS'] += 1
        except NotImplementedYet:
            counts['TODO'] += 1
        except Exception:                           # noqa: BLE001
            counts['FAIL'] += 1
    print(f'passed {counts["PASS"]}   failed {counts["FAIL"]}   '
          f'todo {counts["TODO"]}   skipped {counts["SKIP"]}')
    if counts['TODO']:
        print(f'\n{counts["TODO"]} still TODO. A TODO is not a pass.')


_GATES = {
    '17.1': ('Numerically stable softmax', _g_softmax, False),
    '17.2': ('Cross-entropy gradient', _g_ce_grad, False),
    '19.1': ('Convolution via im2col', _g_conv, False),
    '19.2': ('Convolution output arithmetic', _g_conv_size, False),
    '21.1': ('Scaled dot-product attention', _g_attention, False),
    '18.1': ('A training step that actually trains', _g_training_step, True),
}

print(f'setup complete: {len(_GATES)} exercises loaded, torch={_HAS_TORCH}')

## 17.1 · Numerically stable softmax

*Read first: Ch 17 · PTG p.37*

In [ ]:
def stable_softmax(z: np.ndarray) -> np.ndarray:
    """Softmax over the last axis, overflow-safe. [Ch 17 · PTG p.37]"""
    raise NotImplementedYet('17.1 stable_softmax')

In [ ]:
check('17.1')

## 17.2 · Cross-entropy gradient

*Read first: Ch 17 · PTG p.40*

In [ ]:
def cross_entropy_grad(logits: np.ndarray,
                       one_hot: np.ndarray) -> np.ndarray:
    """Gradient of cross-entropy w.r.t. the logits. [Ch 17 · PTG p.40]"""
    raise NotImplementedYet('17.2 cross_entropy_grad')

In [ ]:
check('17.2')

## 19.1 · Convolution via im2col

*Read first: Ch 19 · PTG p.128*

In [ ]:
def conv2d(x: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Valid single-channel 2-D cross-correlation. [Ch 19 · PTG p.128]"""
    raise NotImplementedYet('19.1 conv2d')

In [ ]:
check('19.1')

## 19.2 · Convolution output arithmetic

*Read first: Ch 19 · PTG p.129*

In [ ]:
def conv_output_size(in_size: int, kernel: int, stride: int,
                     padding: int, dilation: int) -> int:
    """Convolution output size. [Ch 19 · PTG p.129]"""
    raise NotImplementedYet('19.2 conv_output_size')

In [ ]:
check('19.2')

## 21.1 · Scaled dot-product attention

*Read first: Ch 21 · PTG p.320*

In [ ]:
def attention(q: np.ndarray, k: np.ndarray,
              v: np.ndarray) -> np.ndarray:
    """Scaled dot-product attention. [Ch 21 · PTG p.320]"""
    raise NotImplementedYet('21.1 attention')

In [ ]:
check('21.1')

## 18.1 · A training step that actually trains

*Read first: Ch 18 · PTG p.62*

> Needs PyTorch. Colab has it preinstalled.

In [ ]:
def torch_training_step(model, xb, yb, optimiser, loss_fn) -> float:
    """One zero_grad / forward / backward / step. [Ch 18 · PTG p.62]"""
    raise NotImplementedYet('18.1 torch_training_step')

In [ ]:
check('18.1')

---

## Summary

Run this once you have attempted everything above.

In [ ]:
check_all()